In [ ]:
SRC = "/kaggle/input/trendyol-src/src"
DATA = "/kaggle/input/competitions/trendyol-e-ticaret-yarismasi-2026-kaggle"
WORK = "/kaggle/working"

In [ ]:
  !pip install -q rapidfuzz text-unidecode

In [ ]:
  from pathlib import Path
  import shutil, os, sys

  DATA_DIR = Path("/kaggle/input/competitions/trendyol-e-ticaret-yarismasi-2026-kaggle")
  SRC_IN = Path("/kaggle/input/datasets/z3lkahmet/baseline-scripts")
  SRC = Path("/kaggle/working/src")
  WORK = Path("/kaggle/working")

  if not DATA_DIR.exists():
      DATA_DIR = Path("/kaggle/input/trendyol-e-ticaret-yarismasi-2026-kaggle")

  shutil.copytree(SRC_IN, SRC, dirs_exist_ok=True)
  sys.path.insert(0, str(SRC))

  print(DATA_DIR)
  print(SRC)

In [ ]:
 !python /kaggle/working/src/train_term_negatives.py \
    --data-dir {DATA_DIR} \
    --output /kaggle/working/train_term_negatives.csv

In [ ]:
!python /kaggle/working/src/sample_negative_audit.py \
    --data-dir {DATA_DIR} \
    --negatives /kaggle/working/train_term_negatives.csv \
    --output /kaggle/working/negative_audit_sample.csv \
    --n-per-band 200

In [ ]:
  !python /kaggle/working/src/transformer_biencoder.py train \
    --data-dir {DATA_DIR} \
    --negatives /kaggle/working/train_term_negatives.csv \
    --model-dir /kaggle/working/transformer_biencoder \
    --epochs 2 \
    --batch-size 64 \
    --fp16

In [ ]:
  !python /kaggle/working/src/transformer_biencoder.py predict \
    --data-dir {DATA_DIR} \
    --model-dir /kaggle/input/datasets/z3lkahmet/bi-encoder \
    --output /kaggle/working/transformer_biencoder_scores.csv \
    --device cuda

In [ ]:
  !find /kaggle/working -maxdepth 3 -type f | sort | grep -E "score|catboost|lgbm|transformer|csv"
  !find /kaggle/input -maxdepth 4 -type f | sort | grep -E "score|catboost|lgbm|transformer|csv"

In [ ]:
!python /kaggle/working/src/ensemble_scores.py blend \
    --catboost /kaggle/input/datasets/z3lkahmet/outputs/catboost_category_aware_2026-06-30_scores.csv \
    --lgbm /kaggle/input/datasets/z3lkahmet/outputs/pu_lgbm_v1_scores.csv \
    --transformer /kaggle/working/transformer_biencoder_scores.csv \
    --output /kaggle/working/moe_gbdt_rankblend_scores.csv \
    --report-correlations

In [ ]:
  !python /kaggle/working/src/ensemble_scores.py submit \
    --scores /kaggle/working/moe_gbdt_rankblend_scores.csv \
    --output-dir /kaggle/working \
    --rates 0.18 0.20 0.22 0.24

In [ ]:
  !python /kaggle/working/src/ensemble_scores.py submit \
    --scores /kaggle/working/moe_gbdt_rankblend_scores.csv \
    --output-dir /kaggle/working \
    --rates 0.24 0.25 0.26 0.27 0.28 0.30

In [ ]:
  !python src/ensemble_scores.py submit \
    --scores /kaggle/input/datasets/z3lkahmet/outputs/moe_gbdt_rankblend_scores.csv \
    --output-dir /kaggle/working \
    --prefix submission_moe_constrained \
    --rates 0.25 0.26 0.27 \
    --min-positive-excess-base 100

In [ ]:
  for w in [0.55, 0.60, 0.65, 0.70]:
      out = f"/kaggle/working/moe_blend_t{int(w*100)}.csv"
      !python /kaggle/working/src/ensemble_scores.py blend \
        --catboost /kaggle/input/datasets/z3lkahmet/outputs/catboost_category_aware_2026-06-30_scores.csv \
        --lgbm /kaggle/input/datasets/z3lkahmet/outputs/pu_lgbm_v1_scores.csv \
        --transformer /kaggle/working/transformer_biencoder_scores.csv \
        --output {out} \
        --gbdt-weight {1-w} \
        --transformer-weight {w}

In [ ]:
  from pathlib import Path

  score_files = {
      "t55": "/kaggle/input/datasets/z3lkahmet/moe-blends/moe_blend_t55.csv",
      "t60": "/kaggle/input/datasets/z3lkahmet/moe-blends/moe_blend_t60.csv",
      "t65": "/kaggle/input/datasets/z3lkahmet/moe-blends/moe_blend_t65.csv",
      "t70": "/kaggle/input/datasets/z3lkahmet/moe-blends/moe_blend_t70.csv",
  }

  for name, path in score_files.items():
      assert Path(path).exists(), path
      !python /kaggle/working/src/ensemble_scores.py submit \
        --scores {path} \
        --output-dir /kaggle/working \
        --prefix submission_moe_blend_{name} \
        --rates 0.24 0.25 0.26 0.27 0.28

In [ ]:
 !tar -czf /kaggle/working/transformer_biencoder.tar.gz -C /kaggle/working transformer_biencoder

In [ ]:
!ls /kaggle/working

In [ ]:
  !cd /kaggle/working && rm -f working_no_src.zip && zip -r working_no_src.zip . -x "src/*" "./src/*" "working_no_src.zip" && ls -lh
  working_no_src.zip

In [ ]:
!kaggle competitions submit -c trendyol-e-ticaret-yarismasi-2026-kaggle -f /kaggle/working/submission_moe_blend_t65_r28.csv  -m "transformers blend"